# Notebook 01 — Scientific Python Stack

This notebook demonstrates the four libraries you will use in almost every physics
calculation: **NumPy**, **SciPy**, **Pandas**, and **Matplotlib**.
It also shows how to save and reload results with **h5py**, and how to import
your own package.

The running example is the IceCube diffuse muon neutrino spectrum.

> **Package required**: the `neutrino_flux` example package must be installed.  
> From `examples/coding/neutrino_flux/`, run: `pip install -e .`

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from neutrino_flux import PowerLaw, flux_above, threshold_energy


def _find_style() -> Path | None:
    """Search upward from CWD for the nestling.mplstyle file."""
    path = Path.cwd()
    for _ in range(6):
        candidate = path / "examples" / "plotting" / "styles" / "nestling.mplstyle"
        if candidate.exists():
            return candidate
        path = path.parent
    return None


_style = _find_style()
if _style:
    plt.style.use(str(_style))
    print(f"Style loaded: {_style}")
else:
    print("Warning: nestling.mplstyle not found, using default style")

## 1. NumPy — building the spectrum

NumPy arrays apply operations to every element at once (**vectorisation**).
No `for` loop needed.

In [ ]:
# IceCube best-fit single power law (arXiv:2111.10973, Table 2)
model = PowerLaw(gamma=2.37, norm=1.66e-18, E_ref=1e5)

E = np.logspace(3, 7, 500)    # 1 TeV to 10 PeV  [GeV]
phi = model(E)

print(f"Energy range : {E[0]:.1e} – {E[-1]:.1e} GeV")
print(f"Flux at E_ref: {model(1e5):.3e} GeV-1 cm-2 s-1 sr-1")
print(f"Flux array shape: {phi.shape}")

## 2. SciPy — integrating the flux

The `neutrino_flux` package wraps `scipy.integrate.quad` and `scipy.optimize.brentq`
with a log-energy substitution that keeps the integration numerically stable for
power laws spanning many decades.

See `examples/coding/neutrino_flux/neutrino_flux/integrate.py` for the implementation
and a note on why raw `quad` in linear energy space fails here.

In [ ]:
# Total integrated flux from 1 TeV to 10 PeV
total = flux_above(model, E_min=1e3, E_max=1e7)
print(f"Integrated flux (1 TeV – 10 PeV): {total:.3e}  cm-2 s-1 sr-1")

# Integrated flux above 1 TeV (to effective infinity)
total_inf = flux_above(model, E_min=1e3)
print(f"Integrated flux (>1 TeV):          {total_inf:.3e}  cm-2 s-1 sr-1")

# Find the energy at which the cumulative flux drops below a threshold.
# Choose a threshold within the physically reachable range of this spectrum:
#   flux_above(1 TeV)   ≈ 6.6e-11  cm-2 s-1 sr-1
#   flux_above(100 PeV) ≈ 9.4e-18  cm-2 s-1 sr-1
# A threshold of 1e-14 corresponds to an energy threshold of ~600 TeV.
threshold = 1e-14   # cm-2 s-1 sr-1

E_cut = threshold_energy(model, threshold=threshold)
print(f"\nCumulative flux = {threshold:.0e}: E_cut = {E_cut:.3e} GeV  ({E_cut/1e3:.0f} TeV)")

# Verify
check = flux_above(model, E_min=E_cut)
print(f"Verification   flux_above(E_cut) = {check:.3e}  (target {threshold:.1e})")

## 3. Pandas — tabular results

Store the spectrum in a `DataFrame` for easy inspection and export.

In [ ]:
df = pd.DataFrame({
    "energy_GeV":            E,
    "flux_per_GeV_cm2_s_sr": phi,
    "E2_flux":               E**2 * phi,
})

print(df.head())
print("\nSummary statistics:")
print(df[["energy_GeV", "E2_flux"]].describe())

## 4. h5py — saving and reloading

HDF5 is the standard format for physics simulation output.
The `with` statement guarantees the file is flushed and closed even if an error occurs.

In [ ]:
out = Path("/tmp/icecube_spectrum.h5")

with h5py.File(out, "w") as f:
    f.create_dataset("energy", data=E, compression="gzip")
    f.create_dataset("flux",   data=phi, compression="gzip")
    f.attrs["gamma"] = model.gamma
    f.attrs["norm"]  = model.norm
    f.attrs["E_ref"] = model.E_ref
    f.attrs["units"] = "GeV, GeV-1 cm-2 s-1 sr-1"

with h5py.File(out, "r") as f:
    E_loaded   = f["energy"][:]
    phi_loaded = f["flux"][:]
    print("Loaded attributes:", dict(f.attrs))

assert np.allclose(E, E_loaded)
assert np.allclose(phi, phi_loaded)
print("\nRound-trip check passed.")

## 5. Matplotlib — publication-quality plot

The `nestling.mplstyle` loaded above sets font sizes, tick directions,
and a colourblind-friendly colour cycle.  Override `figsize` explicitly
when the default `3 × 3` is too small for a multi-panel figure.

In [ ]:
# Cumulative flux above E (used for right panel)
E_cum = np.logspace(3, 7, 100)
cum   = np.array([flux_above(model, E_min=e) for e in E_cum])

fig, axes = plt.subplots(1, 2, figsize=(7, 3))

# Left: E^2-weighted differential spectrum
ax = axes[0]
ax.loglog(E, E**2 * phi, label=rf"$\gamma = {model.gamma}$")
ax.axvline(E_cut, ls="--", label=rf"$E_{{\rm cut}}$")
ax.set_xlabel(r"$E$ [GeV]")
ax.set_ylabel(r"$E^2\,\Phi$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]")
ax.legend()

# Right: cumulative flux above E
ax = axes[1]
ax.loglog(E_cum, cum)
ax.axhline(threshold, ls="--", label=rf"threshold $= {threshold:.0e}$")
ax.axvline(E_cut,     ls=":",  label=rf"$E_{{\rm cut}} = {E_cut:.1e}$ GeV")
ax.set_xlabel(r"$E_{\rm min}$ [GeV]")
ax.set_ylabel(r"$\Phi(>E_{\rm min})$  [cm$^{-2}$ s$^{-1}$ sr$^{-1}$]")
ax.legend(fontsize=6)

fig.tight_layout()
fig.savefig("/tmp/icecube_spectrum.pdf")
plt.show()
print("Saved to /tmp/icecube_spectrum.pdf")

## Summary

| Library | Task in this notebook |
|---------|----------------------|
| NumPy | Vectorised power-law spectrum on a log-spaced array |
| SciPy (via `neutrino_flux`) | Log-space integration, `brentq` threshold inversion |
| Pandas | Tabular storage, `head`, `describe` |
| h5py | Compressed HDF5 read/write with metadata attributes |
| Matplotlib | Two-panel log-log plot using the nestling style |

See the [Scientific Python Stack lesson](../../../docs/tracks/05-coding/lesson-04.md).